# Inverse Problems by Differentiable FEM

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/camlab-ethz/TensorMesh/blob/main/notebooks/coefficient_identification.ipynb)

Recover an unknown, spatially varying diffusion coefficient $\kappa(x)$ in

$$-\nabla\cdot(\kappa(x)\,\nabla u) = f, \qquad u = 0 \text{ on } \partial\Omega$$

from a **single** observed solution — purely by gradient descent *through*
the FEM solve. There is no adjoint equation and no sensitivity code anywhere
in this notebook: `loss.backward()` differentiates the sparse solve directly,
because the whole pipeline is native PyTorch autograd.

This is the capability that separates TensorMesh from a classical FEM stack.

⏱️ *The optimisation loop takes a few minutes on Colab's free CPU runtime;
lower `N_ITER` below for a quicker (coarser) recovery.*

Docs: [Coefficient-Field Identification](https://docs.tensor-mesh.com/example_gallery/inverse_design.html#coefficient-field-identification-coefficient-identification-py) · Source: [`examples/inverse_design/coefficient_identification.py`](https://github.com/camlab-ethz/TensorMesh/blob/main/examples/inverse_design/coefficient_identification.py)

In [ ]:
# Install TensorMesh (skipped automatically if it is already available, e.g. a local dev setup).
# The apt line provides the OpenGL utility library that gmsh -- TensorMesh's mesh generator --
# needs at import time; it is a no-op where the library is already present.
import importlib.util
if importlib.util.find_spec("tensormesh") is None:
    !apt-get -qq install -y libglu1-mesa > /dev/null 2>&1 || true
    %pip install -q tensormesh-fem==0.2.0

## The forward model

A coefficient-weighted Laplacian. Note that `kappa` enters as `point_data`,
so it is an ordinary tensor — and therefore something autograd can track.

In [ ]:
import time

import matplotlib.pyplot as plt
import matplotlib.tri as mtri
import numpy as np
import torch

from tensormesh import Condenser, ElementAssembler, Mesh, NodeAssembler

torch.set_default_dtype(torch.float64)


class WeightedLaplace(ElementAssembler):
    """Stiffness for ``-div(kappa grad u)`` with a per-node coefficient."""

    def forward(self, gradu, gradv, kappa):
        return kappa * (gradu @ gradv)


class Source(NodeAssembler):
    """Load vector for a nodal source ``f``."""

    def forward(self, v, f):
        return v * f

## Assemble the differentiable forward map

`fem_solve` maps a coefficient field to a solution. Every step inside it —
assembly, condensation, sparse solve — carries gradients.

In [ ]:
H = 0.05        # mesh size
N_ITER = 3000   # Adam steps; the full example uses 5000 for a slightly sharper recovery
LR = 3e-2

mesh = Mesh.gen_rectangle(chara_length=H)
cond = Condenser(mesh.boundary_mask)
x, y = mesh.points[:, 0], mesh.points[:, 1]
print(f"mesh: {mesh.n_points} nodes")

# Ground truth: a smooth, strictly positive four-lobe coefficient field.
kappa_true = 1.0 + 0.6 * torch.sin(2 * torch.pi * x) * torch.cos(2 * torch.pi * y)
f_vals = torch.ones(mesh.n_points)

# Reusable assemblers (topology is fixed; only the coefficient changes).
stiffness = WeightedLaplace.from_mesh(mesh)
load = Source.from_mesh(mesh)


def fem_solve(kappa):
    """The differentiable forward map: assemble -> condense -> solve."""
    K = stiffness(point_data={"kappa": kappa}).double()
    b = load(point_data={"f": f_vals}).double()
    K_, b_ = cond(K, b)
    return cond.recover(K_.solve(b_))


# The single synthetic observation we are allowed to see.
with torch.no_grad():
    u_obs = fem_solve(kappa_true)
u_norm = u_obs.abs().max().item()

## Optimize

Adam on $\theta$, with $\kappa = 1 + \tanh\theta$ keeping the coefficient
positive. The only thing connecting the loss to $\theta$ is the FEM solve
itself.

In [ ]:
# theta is unconstrained; kappa = 1 + tanh(theta) stays in (0, 2), so the FEM
# matrix is SPD at every iteration.
theta = torch.zeros(mesh.n_points, requires_grad=True)
optim = torch.optim.Adam([theta], lr=LR)

losses, u_errs = [], []
t0 = time.time()
for step in range(N_ITER):
    optim.zero_grad()
    kappa = 1.0 + torch.tanh(theta)
    u = fem_solve(kappa)
    loss = ((u - u_obs) ** 2).sum()
    loss.backward()          # <- autograd differentiates through the sparse solve
    optim.step()
    with torch.no_grad():
        losses.append(loss.item())
        u_errs.append(((u - u_obs).abs().max() / u_norm).item())
    if step % 250 == 0 or step == N_ITER - 1:
        print(f"  step {step:4d}  loss={losses[-1]:.3e}  u_rel={u_errs[-1]:.3e}")
print(f"{N_ITER} steps in {time.time() - t0:.1f}s")

with torch.no_grad():
    kappa_rec = 1.0 + torch.tanh(theta)

## Results

In [ ]:
pts = mesh.points.numpy()
triang = mtri.Triangulation(pts[:, 0], pts[:, 1], mesh.cells["triangle"].numpy())
k_true, k_rec = kappa_true.numpy(), kappa_rec.numpy()

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), constrained_layout=True)
levels = np.linspace(0.4, 1.6, 21)
panels = [
    (k_true, r"true $\kappa(x)$", "viridis", levels),
    (k_rec, r"recovered $\kappa(x)$", "viridis", levels),
    (np.abs(k_rec - k_true), r"$|\kappa_{rec} - \kappa_{true}|$", "Reds", 21),
]
for ax, (data, title, cmap, lv) in zip(axes, panels):
    cs = ax.tricontourf(triang, data, levels=lv, cmap=cmap)
    ax.set_aspect("equal")
    ax.set_title(title)
    fig.colorbar(cs, ax=ax, shrink=0.82)
fig.suptitle(f"After {N_ITER} Adam steps: data loss {losses[-1]:.1e}", y=1.04)
plt.show()

fig, ax = plt.subplots(figsize=(6.2, 4))
ax.semilogy(losses, color="#c0392b", lw=2, label=r"data loss $\|u_\theta - u_{obs}\|^2$")
ax.semilogy(u_errs, color="#2980b9", lw=2, ls="--", label="relative max-norm error in $u$")
ax.set_xlabel("Adam iteration")
ax.set_title("Gradient flow through the FEM solve")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

## Where to next

- The same autograd path drives [topology optimization](https://docs.tensor-mesh.com/example_gallery/inverse_design.html) — density fields instead of coefficients.
- [Physics-informed learning](https://colab.research.google.com/github/camlab-ethz/TensorMesh/blob/main/notebooks/poisson_galerkin.ipynb) — put a neural network inside the loop.
- Raise `N_ITER` (the full example uses 5000) for a sharper reconstruction.